# Notebook 05 - Text Preprocessing IndoBERT Dataset

Notebook ini menyiapkan input teks untuk baseline IndoBERT.

Tahap ini **belum melakukan training model**. Output notebook ini akan dipakai oleh:

- Notebook 07: `text_only_baseline_indobert`
- Notebook 09: `late_fusion_multimodal_baseline`
- Notebook 10: `cross_modal_or_gated_fusion_model`

Output utama:

- teks hasil normalisasi ringan;
- `input_ids`, `attention_mask`, dan `token_type_ids`;
- label matrix 7 aspek;
- audit panjang token dan truncation;
- class weight awal untuk training text-only;
- file `.npz` siap dimuat oleh PyTorch pada Notebook 07.

## Dasar Metodologis

Tahap preprocessing teks dibuat **ringan** dan tidak memakai stemming/stopword removal agresif.

Alasan ilmiahnya:

1. **BERT fine-tuning memakai tokenizer dan distribusi pretraining yang spesifik.** BERT dirancang agar downstream task cukup menambahkan output layer dan fine-tuning, sehingga preprocessing sebaiknya tidak mengubah teks secara agresif di luar ekspektasi tokenizer.

2. **IndoBERT dari IndoNLU memang dibuat untuk NLU bahasa Indonesia.** IndoNLU menyediakan benchmark dan model IndoBERT yang dilatih dari korpus Indonesia besar, termasuk teks dari media sosial, blog, berita, dan web. Karena dataset review hotel ini berbahasa Indonesia informal/semi-formal, IndoBERT adalah baseline text encoder yang metodologis.

3. **Subword tokenization membantu menangani variasi kata, typo ringan, dan kata tidak umum.** Karena itu, tokenisasi dilakukan memakai tokenizer bawaan model, bukan tokenisasi manual yang bisa merusak pemetaan subword.

4. **Label MABSA bersifat multi-aspect dan imbalanced.** Karena Notebook 04 menunjukkan dominasi label `None`, Notebook 05 menyimpan class weight dan audit label agar Notebook 07 tidak hanya mengandalkan accuracy.

Referensi:

- Devlin et al. (2019), *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. https://aclanthology.org/N19-1423/
- Wilie et al. (2020), *IndoNLU: Benchmark and Resources for Evaluating Indonesian Natural Language Understanding*. https://aclanthology.org/2020.aacl-main.85/
- Schuster & Nakajima (2012), *Japanese and Korean voice search*, yang memperkenalkan WordPiece untuk segmentasi subword.
- Sokolova & Lapalme (2009), *A systematic analysis of performance measures for classification tasks*. https://doi.org/10.1016/j.ipm.2009.03.002

### 1. Import Library dan Persiapan Direktori Output


In [1]:
from pathlib import Path
import html
import json
import math
import os
import re
import shutil
import subprocess
import sys
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)

SEED = 42
MODEL_NAME = os.environ.get("INDOBERT_MODEL_NAME", "indobenchmark/indobert-base-p1")
MAX_LENGTH = int(os.environ.get("INDOBERT_MAX_LENGTH", "256"))
BATCH_SIZE_TOKENIZE = int(os.environ.get("TOKENIZE_BATCH_SIZE", "256"))

ASPECTS = ["Kamar", "Kebersihan", "Pelayanan", "Harga", "Lokasi", "Fasilitas", "Makanan"]
LABEL_ORDER = ["None", "Negatif", "Netral", "Positif"]
LABEL_TO_ID = {"None": 0, "Negatif": 1, "Netral": 2, "Positif": 3}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}
ASPECT_TO_ID = {aspect: idx for idx, aspect in enumerate(ASPECTS)}
SPLIT_ORDER = ["train", "val", "test"]

IS_KAGGLE = Path("/kaggle").exists()
OUTPUT_DIR = Path("/kaggle/working/mabsa_text_indobert_preprocessing_outputs") if IS_KAGGLE else Path("mabsa_text_indobert_preprocessing_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Kaggle environment:", IS_KAGGLE)
print("Model name:", MODEL_NAME)
print("Max length:", MAX_LENGTH)

Output directory: /kaggle/working/mabsa_text_indobert_preprocessing_outputs
Kaggle environment: True
Model name: indobenchmark/indobert-base-p1
Max length: 256


### 2. Fungsi Pencarian File Dataset


In [2]:
def find_file(filename):
    candidates = []

    if IS_KAGGLE:
        candidates.append(Path("/kaggle/working/mabsa_final_dataset_outputs") / filename)
        input_root = Path("/kaggle/input")
        if input_root.exists():
            candidates.extend(input_root.rglob(filename))

    local_candidates = [
        Path(r"C:\Users\cencen04_\Documents\Codex\2026-05-23\files-mentioned-by-the-user-data\mabsa_final_dataset_outputs") / filename,
        Path(r"G:\Skripsi\Data Preprocessing\mabsa_final_dataset_outputs") / filename,
    ]
    candidates.extend(local_candidates)

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Tidak menemukan {filename}. Jalankan Notebook 03 lebih dulu atau upload output Notebook 03 sebagai Kaggle input."
    )


DATASET_FILES = {
    "all_valid": "final_review_dataset_all_valid.csv",
    "primary_training": "final_review_dataset_primary_training.csv",
    "core_clean": "final_review_dataset_core_clean.csv",
    "relation_aware": "final_review_dataset_relation_aware.csv",
    "contradiction_ablation": "final_review_dataset_contradiction_ablation.csv",
}

paths = {name: find_file(filename) for name, filename in DATASET_FILES.items()}
paths["label_mapping"] = find_file("label_mapping.json")

for name, path in paths.items():
    print(f"{name:24s}: {path}")

all_valid               : /kaggle/input/notebooks/vince0014/03-final-dataset-builder-and-splits/mabsa_final_dataset_outputs/final_review_dataset_all_valid.csv
primary_training        : /kaggle/input/notebooks/vince0014/03-final-dataset-builder-and-splits/mabsa_final_dataset_outputs/final_review_dataset_primary_training.csv
core_clean              : /kaggle/input/notebooks/vince0014/03-final-dataset-builder-and-splits/mabsa_final_dataset_outputs/final_review_dataset_core_clean.csv
relation_aware          : /kaggle/input/notebooks/vince0014/03-final-dataset-builder-and-splits/mabsa_final_dataset_outputs/final_review_dataset_relation_aware.csv
contradiction_ablation  : /kaggle/input/notebooks/vince0014/03-final-dataset-builder-and-splits/mabsa_final_dataset_outputs/final_review_dataset_contradiction_ablation.csv
label_mapping           : /kaggle/input/notebooks/vince0014/03-final-dataset-builder-and-splits/mabsa_final_dataset_outputs/label_mapping.json


### 3. Memuat Dataset dan Konfigurasi Label Mapping


In [3]:
datasets = {name: pd.read_csv(path) for name, path in paths.items() if name in DATASET_FILES}

with open(paths["label_mapping"], "r", encoding="utf-8") as f:
    label_mapping_from_notebook_03 = json.load(f)

all_valid = datasets["all_valid"].copy()

print("Dataset shapes:")
for name, df in datasets.items():
    print(f"- {name:22s}: {df.shape}")

print("Label mapping from Notebook 03:", label_mapping_from_notebook_03)
display(all_valid.head(3))

Dataset shapes:
- all_valid             : (8030, 66)
- primary_training      : (6854, 66)
- core_clean            : (4628, 66)
- relation_aware        : (6837, 66)
- contradiction_ablation: (1176, 66)
Label mapping from Notebook 03: {'label_to_id': {'None': 0, 'Negatif': 1, 'Netral': 2, 'Positif': 3}, 'id_to_label': {'0': 'None', '1': 'Negatif', '2': 'Netral', '3': 'Positif'}, 'aspect_to_id': {'Kamar': 0, 'Kebersihan': 1, 'Pelayanan': 2, 'Harga': 3, 'Lokasi': 4, 'Fasilitas': 5, 'Makanan': 6}}


,ID_Review,Platform,Wilayah,Nama_Hotel,Review_Date,Text_Review,image_count,image_files_sample,dataset_role,split,split_strata,relation_category,relation_strength,policy_bucket,recommended_fusion_strategy,is_excluded_final,primary_training_candidate,core_clean_candidate,relation_aware_candidate,contradiction_ablation_candidate,text_aspects,image_aspects,shared_aspects,hard_contradiction_aspects,soft_disagreement_aspects,text_aspect_count,image_aspect_count,shared_aspect_count,hard_contradiction_count,soft_disagreement_count,relation_overlap_ratio,label_text_Kamar,label_text_id_Kamar,label_image_agg_Kamar,label_image_mode_Kamar,label_image_mode_id_Kamar,label_text_Kebersihan,label_text_id_Kebersihan,label_image_agg_Kebersihan,label_image_mode_Kebersihan,label_image_mode_id_Kebersihan,label_text_Pelayanan,label_text_id_Pelayanan,label_image_agg_Pelayanan,label_image_mode_Pelayanan,label_image_mode_id_Pelayanan,label_text_Harga,label_text_id_Harga,label_image_agg_Harga,label_image_mode_Harga,label_image_mode_id_Harga,label_text_Lokasi,label_text_id_Lokasi,label_image_agg_Lokasi,label_image_mode_Lokasi,label_image_mode_id_Lokasi,label_text_Fasilitas,label_text_id_Fasilitas,label_image_agg_Fasilitas,label_image_mode_Fasilitas,label_image_mode_id_Fasilitas,label_text_Makanan,label_text_id_Makanan,label_image_agg_Makanan,label_image_mode_Makanan,label_image_mode_id_Makanan
0,1,Traveloka,Bandung,Atlantic City Hotel,5/4/2026,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat istirahat, lokasi strat...",1,R1_G1_687bd728.jpg,contradiction_ablation,train,contradiction_ablation__correlated_but_contradictive,correlated_but_contradictive,conflict,contradiction_ablation,contradiction_aware_or_gated_fusion,False,False,False,False,True,Fasilitas|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Fasilitas|Kebersihan,Fasilitas|Kebersihan,Fasilitas,NaN,6,2,2,1,0,0.3333,Positif,3,NaN,NaN,0,Positif,3,Positif,Positif,3,Positif,3,NaN,NaN,0,NaN,0,NaN,NaN,0,Positif,3,NaN,NaN,0,Negatif,1,Positif,Positif,3,Netral,2,NaN,NaN,0
1,10,Traveloka,Bandung,Atlantic City Hotel,1/17/2026,"Diizinkan check-in sebelum jam 12 siang mungkin karena kami dari luar kota, semua staf ramah dan mendapat hadiah voucher makan malam Natal, terima kasih.",2,R10_G1_8debfff9.jpg|R10_G2_b29ac5d6.jpg,relation_aware_training,test,relation_aware_training__uncorrelated_different_aspects,uncorrelated_different_aspects,low,relation_aware_training,relevance_aware_gated_fusion,False,True,False,True,False,Pelayanan,Kebersihan|Lokasi|Makanan,NaN,NaN,NaN,1,3,0,0,0,0.0000,NaN,0,NaN,NaN,0,NaN,0,Positif,Positif,3,Positif,3,NaN,NaN,0,NaN,0,NaN,NaN,0,NaN,0,Positif,Positif,3,NaN,0,NaN,NaN,0,NaN,0,Positif,Positif,3
2,100,Traveloka,Bandung,Atlantic City Hotel,9/24/2022,"Lokasi strategis. Petugas ramah. Sekuriti top abis, ramah dan pintar memarkirkan mobil, walaupun sempit parkiran di basement, jadi mudah. Makanan rating 7/10. Bersih banget kam...",9,R100_G1_3131b00a.jpg|R100_G2_a4e9a23b.jpg|R100_G3_4541d369.jpg|R100_G4_70815ec9.jpg|R100_G5_898c941e.jpg|R100_G6_57cc0cdf.jpg|R100_G7_5a2d3bd7.jpg|R100_G8_ebba5984.jpg|R100_G9_...,core_clean_training,train,core_clean_training__correlated_non_contradictive,correlated_non_contradictive,high,core_clean_training,standard_text_image_fusion,False,True,True,True,False,Harga|Kamar|Kebersihan|Lokasi|Makanan|Pelayanan,Fasilitas|Kamar|Kebersihan|Lokasi|Pelayanan,Kamar|Kebersihan|Lokasi|Pelayanan,NaN,Kamar,6,5,4,0,1,0.5714,Netral,2,Positif,Positif,3,Positif,3,Netral|Positif,Mixed,-1,Positif,3,Positif,Positif,3,Positif,3,NaN,NaN,0,Positif,3,Netral|Positif,Mixed,-1,NaN,0,Positif,Positif,3,Netral,2,NaN,NaN,0


## Kebijakan Normalisasi Teks

Normalisasi yang dilakukan:

- mengubah HTML entity seperti `&quot;` menjadi karakter asli;
- normalisasi Unicode `NFKC`;
- menghapus control character;
- merapikan whitespace.

Normalisasi yang **tidak** dilakukan:

- tidak stemming;
- tidak stopword removal;
- tidak menghapus tanda baca penting;
- tidak lowercasing paksa.

Alasannya: IndoBERT sudah memiliki tokenizer dan embedding subword sendiri. Preprocessing agresif dapat menggeser teks dari distribusi pretraining dan menghilangkan sinyal sentimen.

### 4. Definisi Fungsi Normalisasi Label dan Preprocessing Teks IndoBERT


In [4]:
def normalize_label(value):
    if pd.isna(value):
        return "None"
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null"}:
        return "None"
    return text


def clean_text_for_indobert(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u0000-\u0008\u000B\u000C\u000E-\u001F\u007F]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


for name, df in datasets.items():
    df["Text_Review"] = df["Text_Review"].fillna("").astype(str)
    df["text_clean"] = df["Text_Review"].map(clean_text_for_indobert)
    df["char_len_raw"] = df["Text_Review"].str.len()
    df["char_len_clean"] = df["text_clean"].str.len()
    df["word_len_clean"] = df["text_clean"].str.split().map(len)
    for aspect in ASPECTS:
        col = f"label_text_{aspect}"
        if col in df.columns:
            df[col] = df[col].map(normalize_label)

all_valid = datasets["all_valid"].copy()

text_quality_rows = []
for name, df in datasets.items():
    text_quality_rows.append({
        "dataset_variant": name,
        "rows": int(len(df)),
        "empty_text_clean": int(df["text_clean"].eq("").sum()),
        "duplicate_ID_Review": int(df["ID_Review"].duplicated().sum()),
        "median_word_len": float(df["word_len_clean"].median()),
        "p95_word_len": float(df["word_len_clean"].quantile(0.95)),
        "max_word_len": int(df["word_len_clean"].max()),
    })

text_quality_summary = pd.DataFrame(text_quality_rows)
display(text_quality_summary)
display(all_valid[["ID_Review", "split", "dataset_role", "Text_Review", "text_clean", "word_len_clean"]].head(5))

,dataset_variant,rows,empty_text_clean,duplicate_ID_Review,median_word_len,p95_word_len,max_word_len
0,all_valid,8030,0,0,21.0,98.00,538
1,primary_training,6854,0,0,19.0,81.00,538
2,core_clean,4628,0,0,23.0,91.00,538
3,relation_aware,6837,0,0,19.0,81.00,538
4,contradiction_ablation,1176,0,0,42.0,162.25,529


,ID_Review,split,dataset_role,Text_Review,text_clean,word_len_clean
0,1,train,contradiction_ablation,"Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat istirahat, lokasi strat...","Pesan mendadak hotel di daerah Kota Bandung saat long weekend dan pada penuh, untungnya masih ada kamar tersedia di Hotel Atlantic City. Penyelamat buat istirahat, lokasi strat...",66
1,10,test,relation_aware_training,"Diizinkan check-in sebelum jam 12 siang mungkin karena kami dari luar kota, semua staf ramah dan mendapat hadiah voucher makan malam Natal, terima kasih.","Diizinkan check-in sebelum jam 12 siang mungkin karena kami dari luar kota, semua staf ramah dan mendapat hadiah voucher makan malam Natal, terima kasih.",24
2,100,train,core_clean_training,"Lokasi strategis. Petugas ramah. Sekuriti top abis, ramah dan pintar memarkirkan mobil, walaupun sempit parkiran di basement, jadi mudah. Makanan rating 7/10. Bersih banget kam...","Lokasi strategis. Petugas ramah. Sekuriti top abis, ramah dan pintar memarkirkan mobil, walaupun sempit parkiran di basement, jadi mudah. Makanan rating 7/10. Bersih banget kam...",100
3,1000,val,contradiction_ablation,"sebuah hotel yang sangat unik, dengan tema kuning, kamar, makanan, hiburan yang luar biasa. Keluarga saya sangat puas dengan semua layanan, keramahtamahan, dan fasilitas mereka...","sebuah hotel yang sangat unik, dengan tema kuning, kamar, makanan, hiburan yang luar biasa. Keluarga saya sangat puas dengan semua layanan, keramahtamahan, dan fasilitas mereka...",88
4,1001,train,core_clean_training,Semuanya baik-baik saja! Satu-satunya hal yang tidak memuaskan hanya tentang spa. Saya ingin mencobanya. Jadi saya menelepon nomor itu dan meminta perawatan. Tapi tidak ada yan...,Semuanya baik-baik saja! Satu-satunya hal yang tidak memuaskan hanya tentang spa. Saya ingin mencobanya. Jadi saya menelepon nomor itu dan meminta perawatan. Tapi tidak ada yan...,111


### 5. Validasi Keberadaan Split dan Kolom Aspek


In [5]:
invalid_rows = []
for dataset_name, df in datasets.items():
    for split in SPLIT_ORDER:
        if split not in set(df["split"].unique()):
            invalid_rows.append({"dataset_variant": dataset_name, "issue": "missing_split", "detail": split})

    for aspect in ASPECTS:
        col = f"label_text_{aspect}"
        if col not in df.columns:
            invalid_rows.append({"dataset_variant": dataset_name, "issue": "missing_label_column", "detail": col})
            continue
        invalid_label_df = df.loc[~df[col].isin(LABEL_ORDER), ["ID_Review", col]]
        for row in invalid_label_df.itertuples(index=False):
            invalid_rows.append({
                "dataset_variant": dataset_name,
                "issue": "invalid_label",
                "ID_Review": row[0],
                "detail": f"{col}={row[1]}",
            })

invalid_preprocessing_audit = pd.DataFrame(invalid_rows)
print("Invalid preprocessing audit rows:", len(invalid_preprocessing_audit))
display(invalid_preprocessing_audit.head(20))

Invalid preprocessing audit rows: 0


""


## Load Tokenizer IndoBERT

Notebook mencoba memuat tokenizer dengan urutan berikut:

1. `indobenchmark/indobert-base-p1` dari Hugging Face.
2. Folder tokenizer/model IndoBERT yang sudah ditambahkan sebagai Kaggle input.

Jika Kaggle tidak punya internet dan model belum ditambahkan sebagai input, tambahkan model IndoBERT sebagai Kaggle model/input atau aktifkan internet pada notebook.

### 6. Inisialisasi IndoBERT Tokenizer (AutoTokenizer)


In [6]:
try:
    from transformers import AutoTokenizer
except ImportError:
    if IS_KAGGLE:
        print("transformers belum tersedia. Mencoba install di Kaggle...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
        from transformers import AutoTokenizer
    else:
        raise ImportError(
            "Package transformers belum tersedia di environment lokal. "
            "Notebook ini ditujukan untuk Kaggle; pastikan transformers tersedia saat menjalankan Notebook 05."
        )


def find_local_tokenizer_candidates():
    candidates = []
    roots = []
    if IS_KAGGLE and Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))

    local_roots = [
        Path(r"C:\Users\cencen04_\Downloads"),
        Path(r"G:\Skripsi"),
    ]
    roots.extend([root for root in local_roots if root.exists()])

    marker_files = {"tokenizer_config.json", "vocab.txt", "config.json"}
    for root in roots:
        try:
            for marker in marker_files:
                for path in root.rglob(marker):
                    parent = path.parent
                    text = str(parent).lower()
                    if "indobert" in text or "indonlu" in text or "indobenchmark" in text:
                        candidates.append(parent)
        except Exception as exc:
            print(f"Skip tokenizer scan root {root}: {exc}")

    unique = []
    seen = set()
    for path in candidates:
        if path not in seen:
            unique.append(path)
            seen.add(path)
    return unique


tokenizer_source = MODEL_NAME
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
except Exception as first_error:
    print("Gagal load tokenizer dari Hugging Face:", first_error)
    local_candidates = find_local_tokenizer_candidates()
    print("Local tokenizer candidates:", local_candidates[:10])
    tokenizer = None
    for candidate in local_candidates:
        try:
            tokenizer = AutoTokenizer.from_pretrained(str(candidate), use_fast=True)
            tokenizer_source = str(candidate)
            break
        except Exception as local_error:
            print(f"Gagal load tokenizer dari {candidate}: {local_error}")
    if tokenizer is None:
        raise RuntimeError(
            "Tokenizer IndoBERT belum bisa dimuat. Aktifkan internet Kaggle atau tambahkan model IndoBERT sebagai input."
        )

print("Tokenizer loaded from:", tokenizer_source)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocab size:", getattr(tokenizer, "vocab_size", "unknown"))
print("Model max length:", getattr(tokenizer, "model_max_length", "unknown"))
print("Special tokens:", tokenizer.special_tokens_map)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded from: indobenchmark/indobert-base-p1
Tokenizer class: BertTokenizer
Vocab size: 30521
Model max length: 1000000000000000019884624838656
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


## Audit Panjang Token

Audit ini menentukan seberapa banyak review yang akan terpotong pada `MAX_LENGTH`.

Default `MAX_LENGTH = 256` dipilih sebagai kompromi awal untuk Kaggle T4:

- cukup panjang untuk sebagian besar review hotel;
- lebih ringan untuk training IndoBERT;
- bisa dinaikkan ke `384` jika truncation terlalu tinggi dan VRAM masih cukup.

### 7. Perhitungan Panjang Token Maksimal dan Jumlah Token UNK (Unknown)


In [7]:
def batched(iterable, batch_size):
    for start in range(0, len(iterable), batch_size):
        yield start, iterable[start:start + batch_size]


texts_all_valid = all_valid["text_clean"].tolist()
token_lengths = np.zeros(len(texts_all_valid), dtype=np.int32)
unk_counts = np.zeros(len(texts_all_valid), dtype=np.int32)

unk_token_id = tokenizer.unk_token_id

for start, batch_texts in batched(texts_all_valid, BATCH_SIZE_TOKENIZE):
    encoded_batch = tokenizer(
        batch_texts,
        add_special_tokens=True,
        truncation=False,
        padding=False,
    )
    for offset, input_ids in enumerate(encoded_batch["input_ids"]):
        idx = start + offset
        token_lengths[idx] = len(input_ids)
        if unk_token_id is not None:
            unk_counts[idx] = int(sum(token_id == unk_token_id for token_id in input_ids))

all_valid["token_len_indobert"] = token_lengths
all_valid["unk_count_indobert"] = unk_counts
all_valid["unk_ratio_indobert"] = np.where(token_lengths > 0, unk_counts / token_lengths, 0.0)
all_valid["will_truncate"] = all_valid["token_len_indobert"].gt(MAX_LENGTH)

token_length_summary = pd.DataFrame([
    {
        "metric": "token_len_indobert",
        "count": int(len(all_valid)),
        "mean": round(float(all_valid["token_len_indobert"].mean()), 4),
        "median": round(float(all_valid["token_len_indobert"].median()), 4),
        "p90": round(float(all_valid["token_len_indobert"].quantile(0.90)), 4),
        "p95": round(float(all_valid["token_len_indobert"].quantile(0.95)), 4),
        "p99": round(float(all_valid["token_len_indobert"].quantile(0.99)), 4),
        "max": int(all_valid["token_len_indobert"].max()),
        "truncated_count_at_max_length": int(all_valid["will_truncate"].sum()),
        "truncated_percentage_at_max_length": round(float(all_valid["will_truncate"].mean() * 100), 4),
        "mean_unk_ratio": round(float(all_valid["unk_ratio_indobert"].mean() * 100), 6),
    }
])

token_length_by_split_role = (
    all_valid
    .groupby(["dataset_role", "split"], dropna=False)
    .agg(
        rows=("ID_Review", "count"),
        mean_token_len=("token_len_indobert", "mean"),
        median_token_len=("token_len_indobert", "median"),
        p95_token_len=("token_len_indobert", lambda s: s.quantile(0.95)),
        max_token_len=("token_len_indobert", "max"),
        truncated_count=("will_truncate", "sum"),
        truncated_percentage=("will_truncate", lambda s: s.mean() * 100),
        mean_unk_ratio=("unk_ratio_indobert", lambda s: s.mean() * 100),
    )
    .reset_index()
)
for col in ["mean_token_len", "median_token_len", "p95_token_len", "truncated_percentage", "mean_unk_ratio"]:
    token_length_by_split_role[col] = token_length_by_split_role[col].round(4)

display(token_length_summary)
display(token_length_by_split_role)

,metric,count,mean,median,p90,p95,p99,max,truncated_count_at_max_length,truncated_percentage_at_max_length,mean_unk_ratio
0,token_len_indobert,8030,43.9457,30.0,92.0,126.0,229.0,704,62,0.7721,8.447368


,dataset_role,split,rows,mean_token_len,median_token_len,p95_token_len,max_token_len,truncated_count,truncated_percentage,mean_unk_ratio
0,all_none_control,test,3,8.6667,8.0,12.50,13,0,0.0000,15.9615
1,all_none_control,train,11,16.0909,10.0,33.50,34,0,0.0000,12.2515
2,all_none_control,val,3,4.6667,5.0,5.90,6,0,0.0000,5.5556
3,contradiction_ablation,test,176,69.7330,55.0,166.50,256,0,0.0000,6.8324
4,contradiction_ablation,train,824,78.2354,55.0,225.00,672,28,3.3981,6.8650
5,contradiction_ablation,val,176,75.1705,58.0,173.00,373,3,1.7045,6.5563
6,core_clean_training,test,694,48.8905,35.0,140.70,547,8,1.1527,8.2306
7,core_clean_training,train,3240,44.7448,32.0,119.00,704,19,0.5864,8.2072
8,core_clean_training,val,694,43.2262,31.0,108.35,321,2,0.2882,8.5102
9,relation_aware_training,test,332,23.1295,16.0,58.90,377,1,0.3012,9.6298


### 8. Tokenisasi Batch Teks Menggunakan Padding dan Truncation


In [8]:
encoding = tokenizer(
    texts_all_valid,
    add_special_tokens=True,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
)

input_ids = np.asarray(encoding["input_ids"], dtype=np.int32)
attention_mask = np.asarray(encoding["attention_mask"], dtype=np.int32)
if "token_type_ids" in encoding:
    token_type_ids = np.asarray(encoding["token_type_ids"], dtype=np.int32)
else:
    token_type_ids = np.zeros_like(input_ids, dtype=np.int32)

print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("token_type_ids shape:", token_type_ids.shape)

input_ids shape: (8030, 256)
attention_mask shape: (8030, 256)
token_type_ids shape: (8030, 256)


### 9. Pembuatan Matriks Label Numerik untuk 7 Aspek MABSA


In [9]:
label_matrix = np.zeros((len(all_valid), len(ASPECTS)), dtype=np.int64)
label_text_columns = []

for aspect_idx, aspect in enumerate(ASPECTS):
    col = f"label_text_{aspect}"
    label_text_columns.append(col)
    labels = all_valid[col].map(normalize_label)
    label_matrix[:, aspect_idx] = labels.map(LABEL_TO_ID).astype(np.int64).to_numpy()

label_matrix_df = all_valid[["ID_Review", "split", "dataset_role"] + label_text_columns].copy()
for aspect in ASPECTS:
    label_matrix_df[f"label_id_{aspect}"] = label_matrix[:, ASPECT_TO_ID[aspect]]

invalid_label_ids = int((label_matrix < 0).sum())
print("label_matrix shape:", label_matrix.shape)
print("invalid_label_ids:", invalid_label_ids)
display(label_matrix_df.head(10))

label_matrix shape: (8030, 7)
invalid_label_ids: 0


,ID_Review,split,dataset_role,label_text_Kamar,label_text_Kebersihan,label_text_Pelayanan,label_text_Harga,label_text_Lokasi,label_text_Fasilitas,label_text_Makanan,label_id_Kamar,label_id_Kebersihan,label_id_Pelayanan,label_id_Harga,label_id_Lokasi,label_id_Fasilitas,label_id_Makanan
0,1,train,contradiction_ablation,Positif,Positif,Positif,None,Positif,Negatif,Netral,3,3,3,0,3,1,2
1,10,test,relation_aware_training,None,None,Positif,None,None,None,None,0,0,3,0,0,0,0
2,100,train,core_clean_training,Netral,Positif,Positif,Positif,Positif,None,Netral,2,3,3,3,3,0,2
3,1000,val,contradiction_ablation,Positif,Negatif,Positif,None,Positif,Positif,Positif,3,1,3,0,3,3,3
4,1001,train,core_clean_training,None,None,Positif,None,Positif,None,Positif,0,0,3,0,3,0,3
5,1002,train,core_clean_training,None,None,Positif,None,Positif,None,Positif,0,0,3,0,3,0,3
6,1003,train,core_clean_training,None,None,None,None,Positif,None,Positif,0,0,0,0,3,0,3
7,1004,train,core_clean_training,Positif,None,Positif,None,Positif,Positif,Positif,3,0,3,0,3,3,3
8,1005,train,core_clean_training,None,None,Positif,None,None,None,None,0,0,3,0,0,0,0
9,1006,test,core_clean_training,None,None,Positif,Positif,Positif,Positif,None,0,0,3,3,3,3,0


### 10. Pemetaan Identitas Split dan Pembuatan Mask per Varian Dataset


In [10]:
split_to_id = {"train": 0, "val": 1, "test": 2}
role_values = sorted(all_valid["dataset_role"].dropna().astype(str).unique().tolist())
role_to_id = {role: idx for idx, role in enumerate(role_values)}

all_valid["split_id"] = all_valid["split"].map(split_to_id).astype(np.int64)
all_valid["dataset_role_id"] = all_valid["dataset_role"].map(role_to_id).astype(np.int64)

variant_masks = {
    "all_valid": np.ones(len(all_valid), dtype=bool),
}

for variant_name, variant_df in datasets.items():
    variant_ids = set(variant_df["ID_Review"].astype(str))
    variant_masks[variant_name] = all_valid["ID_Review"].astype(str).isin(variant_ids).to_numpy()

metadata_columns = [
    "ID_Review", "Platform", "Wilayah", "Nama_Hotel", "Review_Date",
    "Text_Review", "text_clean", "split", "split_id", "dataset_role", "dataset_role_id",
    "relation_category", "relation_strength", "policy_bucket",
    "recommended_fusion_strategy", "image_count", "image_files_sample",
    "char_len_raw", "char_len_clean", "word_len_clean",
    "token_len_indobert", "unk_count_indobert", "unk_ratio_indobert", "will_truncate",
] + label_text_columns + [f"label_id_{aspect}" for aspect in ASPECTS]

metadata_columns = [col for col in metadata_columns if col in all_valid.columns]

preprocessed_all_valid = all_valid[metadata_columns].copy()

def save_variant_npz(variant_name, mask):
    path = OUTPUT_DIR / f"indobert_{variant_name}_maxlen{MAX_LENGTH}.npz"
    np.savez_compressed(
        path,
        input_ids=input_ids[mask],
        attention_mask=attention_mask[mask],
        token_type_ids=token_type_ids[mask],
        labels=label_matrix[mask],
        review_ids=all_valid.loc[mask, "ID_Review"].astype(str).to_numpy(),
        split_ids=all_valid.loc[mask, "split_id"].to_numpy(dtype=np.int64),
        dataset_role_ids=all_valid.loc[mask, "dataset_role_id"].to_numpy(dtype=np.int64),
    )
    return path


saved_npz = []
for variant_name, mask in variant_masks.items():
    saved_npz.append(save_variant_npz(variant_name, mask))

print("Saved NPZ files:")
for path in saved_npz:
    print("-", path.name)

Saved NPZ files:
- indobert_all_valid_maxlen256.npz
- indobert_primary_training_maxlen256.npz
- indobert_core_clean_maxlen256.npz
- indobert_relation_aware_maxlen256.npz
- indobert_contradiction_ablation_maxlen256.npz


### 11. Penyusunan Ringkasan Preprocessing per Varian Dataset


In [11]:
variant_summary_rows = []
for variant_name, mask in variant_masks.items():
    variant_df = all_valid.loc[mask]
    variant_summary_rows.append({
        "dataset_variant": variant_name,
        "rows": int(mask.sum()),
        "train": int(variant_df["split"].eq("train").sum()),
        "val": int(variant_df["split"].eq("val").sum()),
        "test": int(variant_df["split"].eq("test").sum()),
        "truncated_count": int(variant_df["will_truncate"].sum()),
        "truncated_percentage": round(float(variant_df["will_truncate"].mean() * 100), 4) if len(variant_df) else 0.0,
        "mean_token_len": round(float(variant_df["token_len_indobert"].mean()), 4) if len(variant_df) else 0.0,
        "p95_token_len": round(float(variant_df["token_len_indobert"].quantile(0.95)), 4) if len(variant_df) else 0.0,
    })

variant_tokenization_summary = pd.DataFrame(variant_summary_rows)
display(variant_tokenization_summary)

,dataset_variant,rows,train,val,test,truncated_count,truncated_percentage,mean_token_len,p95_token_len
0,all_valid,8030,5620,1205,1205,62,0.7721,43.9457,126.00
1,primary_training,6854,4796,1029,1029,31,0.4523,38.3594,106.00
2,core_clean,4628,3240,694,694,29,0.6266,45.1387,120.00
3,relation_aware,6837,4785,1026,1026,31,0.4534,38.4230,106.00
4,contradiction_ablation,1176,824,176,176,31,2.6361,76.5043,207.25


### 12. Perhitungan Bobot Kelas (Class Weights) untuk Penanganan Imbalance


In [12]:
def compute_class_weights(df, split="train"):
    train_df = df.loc[df["split"].eq(split)].copy()
    weights = {}
    counts_table = []
    for aspect in ASPECTS:
        col = f"label_text_{aspect}"
        counts = train_df[col].map(normalize_label).value_counts().reindex(LABEL_ORDER, fill_value=0)
        total = int(counts.sum())
        aspect_weights = {}
        for label in LABEL_ORDER:
            count = int(counts[label])
            # Smoothed inverse-frequency weight. Smoothing avoids infinite weights for absent classes.
            weight = total / (len(LABEL_ORDER) * max(count, 1)) if total else 1.0
            aspect_weights[label] = round(float(weight), 6)
            counts_table.append({
                "aspect": aspect,
                "label": label,
                "train_count": count,
                "class_weight": aspect_weights[label],
            })
        weights[aspect] = aspect_weights
    return weights, pd.DataFrame(counts_table)


primary_train_ids = set(datasets["primary_training"]["ID_Review"].astype(str))
primary_training_from_all_valid = all_valid.loc[all_valid["ID_Review"].astype(str).isin(primary_train_ids)].copy()

class_weights_primary_training, class_weight_table = compute_class_weights(primary_training_from_all_valid, split="train")
display(class_weight_table)

,aspect,label,train_count,class_weight
0,Kamar,None,2453,0.488789
1,Kamar,Negatif,417,2.875300
2,Kamar,Netral,240,4.995833
3,Kamar,Positif,1686,0.711151
4,Kebersihan,None,3224,0.371898
5,Kebersihan,Negatif,255,4.701961
6,Kebersihan,Netral,89,13.471910
7,Kebersihan,Positif,1228,0.976384
8,Pelayanan,None,2521,0.475605
9,Pelayanan,Negatif,167,7.179641


### 13. Penyimpanan Dataset Teks yang Telah Dipreprocessing (CSV)


In [13]:
def save_csv(df, filename):
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


saved_csv = []
saved_csv.append(save_csv(preprocessed_all_valid, "text_preprocessed_all_valid.csv"))
saved_csv.append(save_csv(preprocessed_all_valid.loc[variant_masks["primary_training"]], "text_preprocessed_primary_training.csv"))
saved_csv.append(save_csv(preprocessed_all_valid.loc[variant_masks["core_clean"]], "text_preprocessed_core_clean.csv"))
saved_csv.append(save_csv(preprocessed_all_valid.loc[variant_masks["relation_aware"]], "text_preprocessed_relation_aware.csv"))
saved_csv.append(save_csv(preprocessed_all_valid.loc[variant_masks["contradiction_ablation"]], "text_preprocessed_contradiction_ablation.csv"))
saved_csv.append(save_csv(label_matrix_df, "text_label_matrix_all_valid.csv"))
saved_csv.append(save_csv(text_quality_summary, "text_quality_summary.csv"))
saved_csv.append(save_csv(token_length_summary, "token_length_summary.csv"))
saved_csv.append(save_csv(token_length_by_split_role, "token_length_by_split_role.csv"))
saved_csv.append(save_csv(variant_tokenization_summary, "variant_tokenization_summary.csv"))
saved_csv.append(save_csv(class_weight_table, "class_weights_primary_training_train.csv"))
saved_csv.append(save_csv(invalid_preprocessing_audit, "invalid_text_preprocessing_audit.csv"))

with open(OUTPUT_DIR / "class_weights_primary_training_train.json", "w", encoding="utf-8") as f:
    json.dump(class_weights_primary_training, f, ensure_ascii=False, indent=2)

mappings = {
    "label_to_id": LABEL_TO_ID,
    "id_to_label": ID_TO_LABEL,
    "aspect_to_id": ASPECT_TO_ID,
    "split_to_id": split_to_id,
    "role_to_id": role_to_id,
    "model_name": MODEL_NAME,
    "tokenizer_source": tokenizer_source,
    "max_length": MAX_LENGTH,
}
with open(OUTPUT_DIR / "indobert_text_dataset_mappings.json", "w", encoding="utf-8") as f:
    json.dump(mappings, f, ensure_ascii=False, indent=2)

print("Saved CSV files:")
for path in saved_csv:
    print("-", path.name)
print("Saved mappings and class weights JSON.")

Saved CSV files:
- text_preprocessed_all_valid.csv
- text_preprocessed_primary_training.csv
- text_preprocessed_core_clean.csv
- text_preprocessed_relation_aware.csv
- text_preprocessed_contradiction_ablation.csv
- text_label_matrix_all_valid.csv
- text_quality_summary.csv
- token_length_summary.csv
- token_length_by_split_role.csv
- variant_tokenization_summary.csv
- class_weights_primary_training_train.csv
- invalid_text_preprocessing_audit.csv
Saved mappings and class weights JSON.


### 14. Penyusunan dan Penyimpanan Meta-Summary Preprocessing (JSON)


In [14]:
summary = {
    "model_name": MODEL_NAME,
    "tokenizer_source": tokenizer_source,
    "max_length": MAX_LENGTH,
    "all_valid_rows": int(len(all_valid)),
    "primary_training_rows": int(variant_masks["primary_training"].sum()),
    "core_clean_rows": int(variant_masks["core_clean"].sum()),
    "relation_aware_rows": int(variant_masks["relation_aware"].sum()),
    "contradiction_ablation_rows": int(variant_masks["contradiction_ablation"].sum()),
    "input_ids_shape": list(input_ids.shape),
    "label_matrix_shape": list(label_matrix.shape),
    "token_length_summary": token_length_summary.to_dict(orient="records"),
    "variant_tokenization_summary": variant_tokenization_summary.to_dict(orient="records"),
    "text_quality_summary": text_quality_summary.to_dict(orient="records"),
    "normalization_policy": {
        "html_unescape": True,
        "unicode_nfkc": True,
        "remove_control_characters": True,
        "normalize_whitespace": True,
        "lowercase_forced": False,
        "stemming": False,
        "stopword_removal": False,
    },
    "recommended_next_notebook": "06_image_preprocessing_feature_dataset.ipynb",
}

with open(OUTPUT_DIR / "text_preprocessing_indobert_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

methodology_notes = f"""# Methodology Notes - Notebook 05 Text Preprocessing IndoBERT Dataset

## Tujuan

Notebook 05 menyiapkan input teks untuk IndoBERT tanpa melakukan training. Output utama adalah token arrays, attention mask, label matrix 7 aspek, audit truncation, dan class weight awal.

## Dasar Metodologi

1. Devlin et al. (2019) menunjukkan BERT dapat di-fine-tune untuk downstream task dengan sedikit layer tambahan. Karena itu, preprocessing teks dibuat ringan agar tetap selaras dengan tokenizer/pretraining BERT.
2. Wilie et al. (2020) memperkenalkan IndoNLU dan IndoBERT untuk Indonesian NLU. Karena review hotel pada dataset ini berbahasa Indonesia, IndoBERT menjadi baseline text encoder yang tepat.
3. Tokenizer subword bawaan model dipakai agar variasi kata, typo ringan, dan kata informal tetap bisa dipetakan ke unit subword.
4. Notebook 04 menunjukkan label imbalanced, sehingga Notebook 05 menyimpan class weight untuk Notebook 07.

## Normalisasi

- HTML entity decode: ya
- Unicode NFKC: ya
- Whitespace normalization: ya
- Stemming: tidak
- Stopword removal: tidak
- Forced lowercasing: tidak

## Konfigurasi Tokenisasi

- Model: {MODEL_NAME}
- Tokenizer source: {tokenizer_source}
- Max length: {MAX_LENGTH}
- Label order: None=0, Negatif=1, Netral=2, Positif=3

## Output Untuk Notebook 07

- `indobert_primary_training_maxlen{MAX_LENGTH}.npz`
- `text_preprocessed_primary_training.csv`
- `class_weights_primary_training_train.json`
- `indobert_text_dataset_mappings.json`

"""

with open(OUTPUT_DIR / "methodology_notes_notebook_05.md", "w", encoding="utf-8") as f:
    f.write(methodology_notes)

print(json.dumps(summary, ensure_ascii=False, indent=2)[:5000])

{
  "model_name": "indobenchmark/indobert-base-p1",
  "tokenizer_source": "indobenchmark/indobert-base-p1",
  "max_length": 256,
  "all_valid_rows": 8030,
  "primary_training_rows": 6854,
  "core_clean_rows": 4628,
  "relation_aware_rows": 6837,
  "contradiction_ablation_rows": 1176,
  "input_ids_shape": [
    8030,
    256
  ],
  "label_matrix_shape": [
    8030,
    7
  ],
  "token_length_summary": [
    {
      "metric": "token_len_indobert",
      "count": 8030,
      "mean": 43.9457,
      "median": 30.0,
      "p90": 92.0,
      "p95": 126.0,
      "p99": 229.0,
      "max": 704,
      "truncated_count_at_max_length": 62,
      "truncated_percentage_at_max_length": 0.7721,
      "mean_unk_ratio": 8.447368
    }
  ],
  "variant_tokenization_summary": [
    {
      "dataset_variant": "all_valid",
      "rows": 8030,
      "train": 5620,
      "val": 1205,
      "test": 1205,
      "truncated_count": 62,
      "truncated_percentage": 0.7721,
      "mean_token_len": 43.9457,
      "p

### 15. Verifikasi Akhir (Critical Checks) Hasil Preprocessing


In [15]:
checks = []

checks.append({
    "check": "invalid_preprocessing_audit_rows",
    "value": int(len(invalid_preprocessing_audit)),
    "expected": 0,
})
checks.append({
    "check": "all_valid_rows_match_arrays",
    "value": int(input_ids.shape[0]),
    "expected": int(len(all_valid)),
})
checks.append({
    "check": "input_ids_max_length",
    "value": int(input_ids.shape[1]),
    "expected": int(MAX_LENGTH),
})
checks.append({
    "check": "attention_mask_shape_match",
    "value": list(attention_mask.shape),
    "expected": list(input_ids.shape),
})
checks.append({
    "check": "token_type_ids_shape_match",
    "value": list(token_type_ids.shape),
    "expected": list(input_ids.shape),
})
checks.append({
    "check": "label_matrix_shape_match",
    "value": list(label_matrix.shape),
    "expected": [int(len(all_valid)), int(len(ASPECTS))],
})
checks.append({
    "check": "invalid_label_ids",
    "value": int(invalid_label_ids),
    "expected": 0,
})
checks.append({
    "check": "empty_clean_text",
    "value": int(all_valid["text_clean"].eq("").sum()),
    "expected": 0,
})
checks.append({
    "check": "primary_training_npz_exists",
    "value": int((OUTPUT_DIR / f"indobert_primary_training_maxlen{MAX_LENGTH}.npz").exists()),
    "expected": 1,
})
checks.append({
    "check": "mappings_json_exists",
    "value": int((OUTPUT_DIR / "indobert_text_dataset_mappings.json").exists()),
    "expected": 1,
})

checks_df = pd.DataFrame(checks)
checks_df["status"] = checks_df.apply(lambda row: "OK" if row["value"] == row["expected"] else "NEEDS_REVIEW", axis=1)
checks_df.to_csv(OUTPUT_DIR / "notebook_05_checks.csv", index=False, encoding="utf-8-sig")
display(checks_df)

if (checks_df["status"] == "NEEDS_REVIEW").any():
    print("Notebook 05 selesai dengan catatan. Cek baris NEEDS_REVIEW sebelum Notebook 06/07.")
else:
    print("Notebook 05 selesai. Dataset teks IndoBERT siap untuk Notebook 07 text-only baseline.")

,check,value,expected,status
0,invalid_preprocessing_audit_rows,0,0,OK
1,all_valid_rows_match_arrays,8030,8030,OK
2,input_ids_max_length,256,256,OK
3,attention_mask_shape_match,"[8030, 256]","[8030, 256]",OK
4,token_type_ids_shape_match,"[8030, 256]","[8030, 256]",OK
5,label_matrix_shape_match,"[8030, 7]","[8030, 7]",OK
6,invalid_label_ids,0,0,OK
7,empty_clean_text,0,0,OK
8,primary_training_npz_exists,1,1,OK
9,mappings_json_exists,1,1,OK


Notebook 05 selesai. Dataset teks IndoBERT siap untuk Notebook 07 text-only baseline.


### 16. Pembuatan Arsip ZIP untuk Output Preprocessing Teks


In [16]:
zip_base = OUTPUT_DIR.parent / "mabsa_text_indobert_preprocessing_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)
print("ZIP output:", zip_path)
print("\nFiles in output directory:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)

ZIP output: /kaggle/working/mabsa_text_indobert_preprocessing_outputs.zip

Files in output directory:
- class_weights_primary_training_train.csv
- class_weights_primary_training_train.json
- indobert_all_valid_maxlen256.npz
- indobert_contradiction_ablation_maxlen256.npz
- indobert_core_clean_maxlen256.npz
- indobert_primary_training_maxlen256.npz
- indobert_relation_aware_maxlen256.npz
- indobert_text_dataset_mappings.json
- invalid_text_preprocessing_audit.csv
- methodology_notes_notebook_05.md
- notebook_05_checks.csv
- text_label_matrix_all_valid.csv
- text_preprocessed_all_valid.csv
- text_preprocessed_contradiction_ablation.csv
- text_preprocessed_core_clean.csv
- text_preprocessed_primary_training.csv
- text_preprocessed_relation_aware.csv
- text_preprocessing_indobert_summary.json
- text_quality_summary.csv
- token_length_by_split_role.csv
- token_length_summary.csv
- variant_tokenization_summary.csv
